# Protocolo experimental

En esta sección se construye el protocolo experimental que será utilizado para comparar los modelos x-vector y ECAPA-TDNN en la tarea de verificación de locutor.

La construcción del protocolo parte del DataFrame limpio obtenido en la fase anterior, el cual contiene audios filtrados de acuerdo con duración, frecuencia de muestreo, locutor y género.

## 1. Carga del DataFrame limpio

Primero se carga el archivo de metadatos limpio generado en la etapa anterior.
Este archivo contiene únicamente los audios que cumplen con los criterios mínimos definidos durante la exploración y limpieza.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

In [2]:
metadata_path = "metadata_ciempiess_light_limpio_preliminar.csv"

df_metadata = pd.read_csv(metadata_path)

df_metadata.head()

,audio_id,speaker_id,gender,audio_path,sampling_rate,num_samples,duration_sec,duration_range
0,CMPL_F_32_11ANG_00003,F_32,female,CMPL_F_32_11ANG_00003.flac,16000,52108,3.256750,2 a 5 s
1,CMPL_F_32_11ANG_00019,F_32,female,CMPL_F_32_11ANG_00019.flac,16000,37938,2.371125,2 a 5 s
2,CMPL_F_32_11ANG_00010,F_32,female,CMPL_F_32_11ANG_00010.flac,16000,43859,2.741188,2 a 5 s
3,CMPL_F_32_11ANG_00005,F_32,female,CMPL_F_32_11ANG_00005.flac,16000,32581,2.036313,2 a 5 s
4,CMPL_F_32_11ANG_00013,F_32,female,CMPL_F_32_11ANG_00013.flac,16000,32590,2.036875,2 a 5 s


El DataFrame limpio contiene los metadatos necesarios para construir el protocolo experimental.  
Cada fila representa un archivo de audio y contiene información del locutor, género, duración, frecuencia de muestreo y ruta del archivo.

A partir de este DataFrame se construirán tres subconjuntos principales:

- conjunto de enrolamiento,
- conjunto de prueba,
- conjunto de trials para verificación.

## 2. Definición formal de la tarea experimental

En este proyecto se abordará una tarea de **verificación de locutor**.

La verificación de locutor consiste en determinar si dos señales de voz pertenecen o no al mismo hablante. A diferencia de la identificación de locutor, donde el sistema debe elegir una identidad entre varias posibles, en la verificación la salida es una decisión binaria:

- aceptar que ambos audios pertenecen al mismo locutor,
- rechazar que ambos audios pertenezcan al mismo locutor.

Formalmente, dado un audio de enrolamiento y un audio de prueba, el sistema debe estimar qué tan similares son sus representaciones acústicas.

Sea $x_e$ un audio de enrolamiento y $x_t$ un audio de prueba.

Un modelo de verificación de locutor transforma cada audio en un vector de características o embedding:

$$
f(x_e) = z_e
$$

$$
f(x_t) = z_t
$$

donde $ z_e $ y $ z_t $ son embeddings de dimensión fija.

Posteriormente, se calcula una función de similitud:

$$
s(z_e, z_t)
$$

Si el score de similitud supera cierto umbral $ \tau $, entonces el sistema acepta que los audios pertenecen al mismo locutor:

$$
s(z_e, z_t) \geq \tau
$$

En caso contrario, se rechaza la hipótesis de que ambos audios pertenezcan al mismo locutor.

En el protocolo experimental se construirán pares de comparación llamados **trials**.

Cada trial estará compuesto por:

- un audio de enrolamiento,
- un audio de prueba,
- una etiqueta binaria.

La etiqueta se define como:

$$
y =
\begin{cases}
1, & \text{si ambos audios pertenecen al mismo locutor} \\
0, & \text{si los audios pertenecen a locutores distintos}
\end{cases}
$$

Por lo tanto, existirán dos tipos de trials:

| Tipo de trial | Condición | Etiqueta |
|---|---|---|
| Genuino | Mismo locutor | 1 |
| Impostor | Locutores distintos | 0 |

### Decisión experimental inicial

Para esta primera versión del protocolo se utilizará el siguiente diseño:

1. Para cada locutor se seleccionará **un audio de enrolamiento**.
2. Los audios restantes de cada locutor se utilizarán como **audios de prueba**.
3. Se construirán trials genuinos comparando cada audio de prueba contra el audio de enrolamiento del mismo locutor.
4. Se construirán trials impostores comparando cada audio de prueba contra el audio de enrolamiento de un locutor distinto.
5. Para evitar que la tarea sea artificialmente sencilla, los trials impostores se construirán preferentemente entre locutores del mismo género.
6. Se utilizará una semilla aleatoria fija para garantizar reproducibilidad.

Este protocolo será utilizado tanto para el modelo x-vector como para ECAPA-TDNN, de forma que ambos modelos sean evaluados bajo exactamente las mismas condiciones.

### Justificación del protocolo

El uso del mismo protocolo para ambos modelos es fundamental para realizar una comparación justa.

Si x-vector y ECAPA-TDNN se evaluaran con diferentes pares de audios, entonces las diferencias observadas en las métricas podrían deberse al protocolo y no necesariamente al desempeño de los modelos.

Por esta razón, primero se construye un archivo de trials independiente del modelo. Posteriormente, ambos sistemas calcularán scores sobre los mismos pares de comparación.

## 3. Preparación del DataFrame para el protocolo

Antes de construir los conjuntos de enrolamiento, prueba y trials, se crea una copia del DataFrame limpio.

Esta copia permitirá trabajar de forma controlada sin modificar el DataFrame original. Además, se fija una semilla aleatoria para garantizar que la selección de audios sea reproducible.

La reproducibilidad es importante porque el protocolo experimental debe poder reconstruirse exactamente en ejecuciones posteriores.

In [12]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

Se utiliza `RANDOM_STATE = 42` como semilla aleatoria.  
Esto garantiza que las selecciones aleatorias realizadas durante la construcción del protocolo puedan reproducirse.

In [20]:
# Creamos una copio del df original para poder recuperarlo si se requiere posteriormente
df_protocol = df_metadata.copy()


In [19]:
# Ordenamos el df por locutor y audio 
df_protocol = (
    df_protocol
    .sort_values(["speaker_id", "audio_id"])
    .reset_index(drop=True)
)

## 4. Construcción del conjunto de enrolamiento

En una tarea de verificación de locutor, el conjunto de enrolamiento contiene las muestras de referencia de cada locutor. Estas muestras representan la voz conocida del hablante. Posteriormente, los audios de prueba serán comparados contra estas referencias para determinar si pertenecen o no al mismo locutor.

Para esta primera versión del protocolo experimental se seleccionará **un audio de enrolamiento por cada locutor**. La selección se realizará de forma aleatoria, pero utilizando una semilla fija para garantizar reproducibilidad.

In [23]:
df_enrollment = (
    df_protocol
    .groupby("speaker_id", group_keys=False)
    .sample(n=1, random_state=RANDOM_STATE)
    .sort_values("speaker_id")
    .reset_index(drop=True)
)

df_enrollment["split"] = "enrollment"

df_enrollment.head()

,audio_id,speaker_id,gender,audio_path,sampling_rate,num_samples,duration_sec,duration_range,split
0,CMPL_F_01_12MAB_00651,F_01,female,CMPL_F_01_12MAB_00651.flac,16000,90036,5.627250,5 a 10 s,enrollment
1,CMPL_F_02_02MAB_00241,F_02,female,CMPL_F_02_02MAB_00241.flac,16000,56244,3.515250,2 a 5 s,enrollment
2,CMPL_F_03_09MAB_00142,F_03,female,CMPL_F_03_09MAB_00142.flac,16000,120857,7.553562,5 a 10 s,enrollment
3,CMPL_F_04_12ANG_00215,F_04,female,CMPL_F_04_12ANG_00215.flac,16000,33306,2.081625,2 a 5 s,enrollment
4,CMPL_F_05_01CAR_00021,F_05,female,CMPL_F_05_01CAR_00021.flac,16000,89708,5.606750,5 a 10 s,enrollment


### Conclusiones del conjunto de enrolamiento

Se construyó el conjunto de enrolamiento seleccionando un audio por cada locutor disponible en el conjunto limpio.

El conjunto resultante contiene **87 audios**, correspondientes a **87 locutores únicos**.  
Cada locutor aparece exactamente una vez en el conjunto de enrolamiento.

Estos audios serán utilizados como referencias de voz durante la etapa de evaluación.  
Posteriormente, cada audio de prueba será comparado contra audios de este conjunto para construir trials genuinos e impostores.

## 5. Construcción del conjunto de prueba

Después de seleccionar un audio de enrolamiento por cada locutor, los audios restantes se utilizarán como conjunto de prueba.

El conjunto de prueba contiene las muestras que serán comparadas contra los audios de enrolamiento.  
En los trials genuinos, cada audio de prueba se comparará contra el audio de enrolamiento del mismo locutor.  
En los trials impostores, cada audio de prueba se comparará contra el audio de enrolamiento de un locutor distinto.

Esta separación garantiza que un mismo archivo de audio no sea utilizado simultáneamente como referencia y como prueba.

In [33]:
enrollment_audio_ids = set(df_enrollment["audio_id"])

len(enrollment_audio_ids)

87

In [34]:
df_test = df_protocol[
    ~df_protocol["audio_id"].isin(enrollment_audio_ids)
].copy()

df_test = df_test.reset_index(drop=True)
df_test["split"] = "test"

df_test.head()

,audio_id,speaker_id,gender,audio_path,sampling_rate,num_samples,duration_sec,duration_range,split
0,CMPL_F_32_11ANG_00003,F_32,female,CMPL_F_32_11ANG_00003.flac,16000,52108,3.256750,2 a 5 s,test
1,CMPL_F_32_11ANG_00019,F_32,female,CMPL_F_32_11ANG_00019.flac,16000,37938,2.371125,2 a 5 s,test
2,CMPL_F_32_11ANG_00010,F_32,female,CMPL_F_32_11ANG_00010.flac,16000,43859,2.741188,2 a 5 s,test
3,CMPL_F_32_11ANG_00005,F_32,female,CMPL_F_32_11ANG_00005.flac,16000,32581,2.036313,2 a 5 s,test
4,CMPL_F_32_11ANG_00013,F_32,female,CMPL_F_32_11ANG_00013.flac,16000,32590,2.036875,2 a 5 s,test


In [36]:
# Verificamos que no haya traslape entre enrollment y test
overlap_audio_ids = set(df_enrollment["audio_id"]).intersection(
    set(df_test["audio_id"])
)

len(overlap_audio_ids)

0

In [39]:
test_audios_per_speaker = (
    df_test
    .groupby("speaker_id")
    .size()
    .sort_values(ascending=False)
)

# Verificamos que cada locutor tenga al menos un audio de prueba
(test_audios_per_speaker >= 1).all()

True

### Conclusiones del conjunto de prueba

Se construyó el conjunto de prueba utilizando todos los audios que no fueron seleccionados como enrolamiento.

El conjunto resultante contiene **13,165 audios de prueba**.  
No existe traslape entre los audios de enrolamiento y los audios de prueba, por lo que cada archivo pertenece exclusivamente a uno de los dos conjuntos.

Además, todos los locutores tienen al menos un audio en el conjunto de prueba.  
Esto permite construir trials genuinos para cada locutor del protocolo.

## 6. Construcción de trials genuinos

Los trials genuinos son pares de comparación donde el audio de enrolamiento y el audio de prueba pertenecen al mismo locutor.

En este protocolo, cada locutor tiene un único audio de enrolamiento. Por lo tanto, cada audio de prueba se compara contra el audio de enrolamiento correspondiente al mismo `speaker_id`.

Estos trials representan casos positivos de verificación de locutor y reciben la etiqueta:

$$
label = 1
$$

Cada trial genuino tendrá la siguiente estructura:

- identificador del locutor de enrolamiento,
- identificador del audio de enrolamiento,
- ruta del audio de enrolamiento,
- identificador del locutor de prueba,
- identificador del audio de prueba,
- ruta del audio de prueba,
- género del locutor,
- etiqueta igual a 1.

In [ ]:
# Primero construiremos una estructura auxiliar para acceder rápidamente al audio de enrolamiento de cada locutor.
enrollment_by_speaker = (
    df_enrollment
    .set_index("speaker_id")
    .to_dict(orient="index")
)

In [42]:
genuine_trials = []

for _, test_row in df_test.iterrows():
    speaker_id = test_row["speaker_id"]
    enrollment_row = enrollment_by_speaker[speaker_id]

    genuine_trials.append({
        "enrollment_speaker_id": speaker_id,
        "enrollment_audio_id": enrollment_row["audio_id"],
        "enrollment_audio_path": enrollment_row["audio_path"],
        "test_speaker_id": speaker_id,
        "test_audio_id": test_row["audio_id"],
        "test_audio_path": test_row["audio_path"],
        "gender_enrollment": enrollment_row["gender"],
        "gender_test": test_row["gender"],
        "trial_type": "genuine",
        "same_speaker": True,
        "same_gender": enrollment_row["gender"] == test_row["gender"],
        "label": 1
    })

In [43]:
df_genuine_trials = pd.DataFrame(genuine_trials)

df_genuine_trials.head()

,enrollment_speaker_id,enrollment_audio_id,enrollment_audio_path,test_speaker_id,test_audio_id,test_audio_path,gender_enrollment,gender_test,trial_type,same_speaker,same_gender,label
0,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00003,CMPL_F_32_11ANG_00003.flac,female,female,genuine,True,True,1
1,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00019,CMPL_F_32_11ANG_00019.flac,female,female,genuine,True,True,1
2,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00010,CMPL_F_32_11ANG_00010.flac,female,female,genuine,True,True,1
3,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00005,CMPL_F_32_11ANG_00005.flac,female,female,genuine,True,True,1
4,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00013,CMPL_F_32_11ANG_00013.flac,female,female,genuine,True,True,1


### Conclusiones de los trials genuinos

Se construyeron los trials genuinos comparando cada audio de prueba contra el audio de enrolamiento correspondiente al mismo locutor.

Como el conjunto de prueba contiene **13,165 audios**, se generaron **13,165 trials genuinos**.

Todos los trials genuinos cumplen que:

- el locutor de enrolamiento y el locutor de prueba son el mismo,
- la etiqueta asignada es igual a 1,
- no existen comparaciones de un audio contra sí mismo,
- el género del enrolamiento y del test coincide.

Estos trials representan los casos positivos del protocolo de verificación de locutor.

## 7. Construcción de trials impostores

Los trials impostores son pares de comparación donde el audio de enrolamiento y el audio de prueba pertenecen a locutores distintos.

Estos trials representan casos negativos de verificación de locutor y reciben la etiqueta:

$$
label = 0
$$

Para esta primera versión del protocolo se construirá un trial impostor por cada audio de prueba, con el objetivo de balancear el número de trials genuinos e impostores.

Además, los impostores se seleccionarán entre locutores del mismo género cuando sea posible. Esta decisión evita que la comparación sea artificialmente sencilla debido a diferencias evidentes entre voces masculinas y femeninas.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# Como queremos seleccionar impostores del mismo género, agrupamos los audios de enrolamiento por género
enrollment_by_gender = {
    gender: group[[
        "speaker_id",
        "audio_id",
        "audio_path",
        "gender"
    ]].to_dict("records")
    for gender, group in df_enrollment.groupby("gender")
}

dict_keys(['female', 'male'])

In [53]:
impostor_trials = []

for _, test_row in df_test.iterrows():
    test_speaker_id = test_row["speaker_id"]
    test_gender = test_row["gender"]

    candidates = [
        enrollment_row
        for enrollment_row in enrollment_by_gender[test_gender]
        if enrollment_row["speaker_id"] != test_speaker_id
    ]

    chosen_enrollment = candidates[
        rng.integers(0, len(candidates))
    ]

    impostor_trials.append({
        "enrollment_speaker_id": chosen_enrollment["speaker_id"],
        "enrollment_audio_id": chosen_enrollment["audio_id"],
        "enrollment_audio_path": chosen_enrollment["audio_path"],
        "test_speaker_id": test_speaker_id,
        "test_audio_id": test_row["audio_id"],
        "test_audio_path": test_row["audio_path"],
        "gender_enrollment": chosen_enrollment["gender"],
        "gender_test": test_gender,
        "trial_type": "impostor",
        "same_speaker": False,
        "same_gender": chosen_enrollment["gender"] == test_gender,
        "label": 0
    })

In [54]:
df_impostor_trials = pd.DataFrame(impostor_trials)

df_impostor_trials.head()

,enrollment_speaker_id,enrollment_audio_id,enrollment_audio_path,test_speaker_id,test_audio_id,test_audio_path,gender_enrollment,gender_test,trial_type,same_speaker,same_gender,label
0,F_03,CMPL_F_03_09MAB_00142,CMPL_F_03_09MAB_00142.flac,F_32,CMPL_F_32_11ANG_00003,CMPL_F_32_11ANG_00003.flac,female,female,impostor,False,True,0
1,F_26,CMPL_F_26_14MAB_00029,CMPL_F_26_14MAB_00029.flac,F_32,CMPL_F_32_11ANG_00019,CMPL_F_32_11ANG_00019.flac,female,female,impostor,False,True,0
2,F_22,CMPL_F_22_06ALX_00053,CMPL_F_22_06ALX_00053.flac,F_32,CMPL_F_32_11ANG_00010,CMPL_F_32_11ANG_00010.flac,female,female,impostor,False,True,0
3,F_15,CMPL_F_15_06ANG_00085,CMPL_F_15_06ANG_00085.flac,F_32,CMPL_F_32_11ANG_00005,CMPL_F_32_11ANG_00005.flac,female,female,impostor,False,True,0
4,F_15,CMPL_F_15_06ANG_00085,CMPL_F_15_06ANG_00085.flac,F_32,CMPL_F_32_11ANG_00013,CMPL_F_32_11ANG_00013.flac,female,female,impostor,False,True,0


### Conclusiones de los trials impostores

Se construyeron los trials impostores comparando cada audio de prueba contra el audio de enrolamiento de un locutor distinto.

Para mantener balanceado el protocolo, se generó un trial impostor por cada audio de prueba.  
Como el conjunto de prueba contiene **13,165 audios**, se generaron **13,165 trials impostores**.

Todos los trials impostores cumplen que:

- el locutor de enrolamiento y el locutor de prueba son distintos,
- la etiqueta asignada es igual a 0,
- no existen comparaciones de un audio contra sí mismo,
- el locutor impostor fue seleccionado dentro del mismo género.

Estos trials representan los casos negativos del protocolo de verificación de locutor.

## 8. Construcción del DataFrame final de trials

Una vez construidos los trials genuinos e impostores, ambos conjuntos se unen en un único DataFrame llamado `df_trials`.

Este DataFrame representa el protocolo experimental final para la tarea de verificación de locutor.

Cada fila corresponde a un par de comparación compuesto por:

- un audio de enrolamiento,
- un audio de prueba,
- una etiqueta binaria,
- información auxiliar sobre locutor, género y tipo de trial.

Este protocolo será utilizado posteriormente para calcular scores de similitud con los modelos x-vector y ECAPA-TDNN.

In [62]:
df_trials = pd.concat(
    [df_genuine_trials, df_impostor_trials],
    ignore_index=True
)

df_trials.head()

,enrollment_speaker_id,enrollment_audio_id,enrollment_audio_path,test_speaker_id,test_audio_id,test_audio_path,gender_enrollment,gender_test,trial_type,same_speaker,same_gender,label
0,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00003,CMPL_F_32_11ANG_00003.flac,female,female,genuine,True,True,1
1,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00019,CMPL_F_32_11ANG_00019.flac,female,female,genuine,True,True,1
2,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00010,CMPL_F_32_11ANG_00010.flac,female,female,genuine,True,True,1
3,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00005,CMPL_F_32_11ANG_00005.flac,female,female,genuine,True,True,1
4,F_32,CMPL_F_32_11ANG_00011,CMPL_F_32_11ANG_00011.flac,F_32,CMPL_F_32_11ANG_00013,CMPL_F_32_11ANG_00013.flac,female,female,genuine,True,True,1


In [63]:
# Los mezclamos para que no queden los genuinos primeor y los impostores despues
df_trials = (
    df_trials
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

df_trials.head()

,enrollment_speaker_id,enrollment_audio_id,enrollment_audio_path,test_speaker_id,test_audio_id,test_audio_path,gender_enrollment,gender_test,trial_type,same_speaker,same_gender,label
0,M_35,CMPL_M_35_15ALX_00068,CMPL_M_35_15ALX_00068.flac,M_03,CMPL_M_03_08MAB_00863,CMPL_M_03_08MAB_00863.flac,male,male,impostor,False,True,0
1,M_01,CMPL_M_01_13ANG_01532,CMPL_M_01_13ANG_01532.flac,M_01,CMPL_M_01_03ANG_00390,CMPL_M_01_03ANG_00390.flac,male,male,genuine,True,True,1
2,M_41,CMPL_M_41_14MAB_00025,CMPL_M_41_14MAB_00025.flac,M_41,CMPL_M_41_14MAB_00005,CMPL_M_41_14MAB_00005.flac,male,male,genuine,True,True,1
3,M_04,CMPL_M_04_02ANG_00433,CMPL_M_04_02ANG_00433.flac,M_04,CMPL_M_04_02ALX_00160,CMPL_M_04_02ALX_00160.flac,male,male,genuine,True,True,1
4,M_26,CMPL_M_26_03ALX_00015,CMPL_M_26_03ALX_00015.flac,M_35,CMPL_M_35_05MAB_00044,CMPL_M_35_05MAB_00044.flac,male,male,impostor,False,True,0


In [64]:
# Agregamos una columna trial_id para identificar cada comparación de forma única.
df_trials.insert(
    0,
    "trial_id",
    [f"trial_{i:06d}" for i in range(len(df_trials))]
)

df_trials.head()

,trial_id,enrollment_speaker_id,enrollment_audio_id,enrollment_audio_path,test_speaker_id,test_audio_id,test_audio_path,gender_enrollment,gender_test,trial_type,same_speaker,same_gender,label
0,trial_000000,M_35,CMPL_M_35_15ALX_00068,CMPL_M_35_15ALX_00068.flac,M_03,CMPL_M_03_08MAB_00863,CMPL_M_03_08MAB_00863.flac,male,male,impostor,False,True,0
1,trial_000001,M_01,CMPL_M_01_13ANG_01532,CMPL_M_01_13ANG_01532.flac,M_01,CMPL_M_01_03ANG_00390,CMPL_M_01_03ANG_00390.flac,male,male,genuine,True,True,1
2,trial_000002,M_41,CMPL_M_41_14MAB_00025,CMPL_M_41_14MAB_00025.flac,M_41,CMPL_M_41_14MAB_00005,CMPL_M_41_14MAB_00005.flac,male,male,genuine,True,True,1
3,trial_000003,M_04,CMPL_M_04_02ANG_00433,CMPL_M_04_02ANG_00433.flac,M_04,CMPL_M_04_02ALX_00160,CMPL_M_04_02ALX_00160.flac,male,male,genuine,True,True,1
4,trial_000004,M_26,CMPL_M_26_03ALX_00015,CMPL_M_26_03ALX_00015.flac,M_35,CMPL_M_35_05MAB_00044,CMPL_M_35_05MAB_00044.flac,male,male,impostor,False,True,0


### Conclusiones del DataFrame final de trials

Se construyó el DataFrame final `df_trials` uniendo los trials genuinos e impostores.

El protocolo final contiene **26,330 trials**, distribuidos de forma balanceada:

- **13,165 trials genuinos** con etiqueta 1,
- **13,165 trials impostores** con etiqueta 0.

Cada trial tiene un identificador único `trial_id`, lo cual permitirá rastrear cada comparación durante la etapa de extracción de embeddings, cálculo de scores y evaluación.

Además, se verificó que no existan comparaciones de un audio contra sí mismo y que las etiquetas sean consistentes con el tipo de trial.

## . Guardado de los archivos del protocolo experimental

Una vez construido y validado el protocolo experimental, se guardan los archivos principales en disco.

Los archivos generados serán:

- `enrollment.csv`: contiene los audios de referencia de cada locutor.
- `test.csv`: contiene los audios utilizados como prueba.
- `trials.csv`: contiene los pares de comparación del protocolo.

Estos archivos permitirán reutilizar exactamente el mismo protocolo para evaluar los modelos x-vector y ECAPA-TDNN.

In [74]:
from pathlib import Path
import json

In [79]:
protocol_dir = Path("protocolo")
protocol_dir.mkdir(parents=True, exist_ok=True)

protocol_dir

WindowsPath('protocolo')

In [80]:
df_enrollment.to_csv(
    protocol_dir / "enrollment.csv",
    index=False
)

In [81]:
df_test.to_csv(
    protocol_dir / "test.csv",
    index=False
)

In [82]:
df_trials.to_csv(
    protocol_dir / "trials.csv",
    index=False
)